# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Khởi tạo môi trường Colab
Cell này clone repo nếu cần, cài dependencies và đồng bộ module `student/` từ bundle cục bộ. Cập nhật bundle sau mỗi thay đổi mã nguồn.


In [ ]:
# Idempotent Colab bootstrap. Clones only if the destination is absent; never resets it.
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

_REPO_URL = 'https://github.com/JJayzdev/K4-Track-4-Day10-VisionLanguageModel.git'
_REPO_DIR = Path('/content/K4-Track-4-Day10-VisionLanguageModel')
if not _REPO_DIR.exists():
    subprocess.run(['git', 'clone', _REPO_URL, str(_REPO_DIR)], check=True)
elif not _REPO_DIR.is_dir():
    raise NotADirectoryError(f'Colab repo path exists but is not a directory: {_REPO_DIR}')

_requirements = _REPO_DIR / 'requirements-colab.txt'
if not (_REPO_DIR / 'scripts' / 'score.py').is_file() or not _requirements.is_file():
    raise RuntimeError(
        f'Existing checkout at {_REPO_DIR} is incomplete; inspect it in Colab before continuing.'
    )
os.chdir(_REPO_DIR)
if str(_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(_REPO_DIR))
print('working directory:', Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(_requirements)], check=True)

for _package in ('torch', 'transformers', 'peft', 'bitsandbytes'):
    try:
        _version = importlib.metadata.version(_package)
    except importlib.metadata.PackageNotFoundError:
        _version = 'not installed'
    print(f'{_package}: {_version}')
try:
    import torch
    _cuda = torch.cuda.is_available()
    print('CUDA available:', _cuda)
    print('GPU:', torch.cuda.get_device_name(0) if _cuda else 'unavailable')
except Exception as _error:
    print('GPU check failed:', type(_error).__name__)
print('Colab initialization complete')

# Self-contained sync cell for the current local student/ implementation.
# It contains source files only: no dataset, secrets, adapters, or model weights.
import base64
import importlib
import importlib.metadata
import io
import json
import os
from pathlib import Path
import sys
import zipfile

# Colab may start in /content while the clone and data live in its child directory.
_content = Path('/content')
_search_roots = [Path.cwd(), _content / 'K4-Track-4-Day10-VisionLanguageModel']
if _content.is_dir():
    _search_roots.extend(p for p in _content.glob('*') if p.is_dir())
_prepared = []
_checkouts = []
_seen = set()
for _path in _search_roots:
    try:
        _resolved = _path.resolve()
    except OSError:
        continue
    if _resolved in _seen:
        continue
    _seen.add(_resolved)
    _has_scripts = (_resolved / 'scripts' / 'score.py').is_file()
    _has_requirements = (_resolved / 'requirements-colab.txt').is_file()
    _has_meta = (_resolved / 'data' / 'splits' / 'meta.json').is_file()
    if _has_scripts and _has_requirements:
        _checkouts.append(_resolved)
        if _has_meta:
            _prepared.append(_resolved)
if len(_prepared) == 1:
    _root = _prepared[0]
elif len(_prepared) > 1:
    raise RuntimeError(f'Multiple prepared project roots found: {_prepared}')
else:
    if len(_checkouts) != 1:
        _diagnostics = []
        if _content.is_dir():
            for _path in sorted(_content.glob('*')):
                if _path.is_dir():
                    _diagnostics.append({
                        'path': str(_path),
                        'has_score_script': (_path / 'scripts' / 'score.py').is_file(),
                        'has_requirements': (_path / 'requirements-colab.txt').is_file(),
                        'has_dataset_meta': (_path / 'data' / 'splits' / 'meta.json').is_file(),
                    })
        raise RuntimeError(f'Could not identify one project checkout. /content markers: {_diagnostics}')
    _root = _checkouts[0]
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_payload = (
    'UEsDBBQAAAAIAKcKTl0WreehLgAAAC4AAAATAAAAc3R1ZGVudC9fX2luaXRfXy5weVNSUgouKU1JzStRyMwtyEnNBbISSzLz84oV0vKLFHwSkxRcEisVDA30lJSUuABQSwMEFAAAAAgAE3dOXbHFM63bBgAAqRQAAA4AAABzdHVkZW50L2NsaS5weeVYW2/bNhR+96/g+CR1lpJ03TAY1YB0TboARdAVQV+ygKAt2mYjiSpJxUmz/PedQ+pCO3KaoXubHpxIPDeey3cOSSn9XZUlr/KkkJUgorL6jtRKVtaQpdLE2CaHj8RqLitZrQiQklqLXC6sVFVKKZ0stSoJY8vGNlowRmRZK22BslKWI5WZTLpvelVzbUT3/tmoyvPX3K4LOe+YP8Brz3XHy2LiydKVVk2VoyXtYqF4zkxdSDtFw0C8YJ21TKuNaRlLlYtil+/LRlTTbj9TooWxCvgXa7G4dl6Y+p1PJpNcLHFdFTewrpQGpQteiEiLLw2wiXwGztLkb3KuKgGimmpGvNSvQitm1srOyFypgmTklBdGxCT5DVlmEwKPEwZLvTgC3o9opXR5dHh4SIlcDoKIAH5Uka6EjWhgDp2SgSeOnWjg9NIhHkRW5H6gAOpa3oqCPngr8IH9gvBPvGjEidZKR0sqqxteyJw4RbLiVrQSQdrgqxm5d19/0A/Ua9YCUqLytK0LWQHcxgYujjD0Mxdx5xL8x1uzgGSTOdKDYwwETeSOOF0Vag7b7kUkL2g8JdfiLit4Oc85qWfESU4rXorUZUdEExpfJkdXg1fQH4OOXQ+cykKcK3uK+dY5olIk1AoB1mIBKXNHgEhoco/m7ex+0IDaWzeUkFQRVMPNjBTS2EvIg6s2d2Cv+Mf5Av/xdrmy0bDWlVB6rFdNCbX5wa1EuTALLWust4yxXC0Y82aYZg5snj/lOSRKM/dvBplsBvnjIIBOXfbBlvLsQjfCs9sand/MHavni6grCtoRuCXemgN+ThaqWsrVPnmPyJ20ZKm5wxRgs3e1yJZQoVA94C3eFDY7Sg/3CtC8uu7YIDBP2AX5C4SLtZILYbLLkUq4GlQGlbRPompsAknw7K26VDQtS68HsoMf+KVWFR/zOm9yab23gHstijqjtZYdPJOCz0VBvjRQq/aObKRdg3kO6BxyEweBvYLvsO2/dqwc222Lyi2Jlwxk0msvodvworhj4nZRNEYCKmNrqKOROHjex0b3ldx7Myjuc3igBRZY1gDF4rYWWiLrUPT0SfGI1gmiNUj3mZ1R314sWBYE0G3ThYvYtQBlcgUYW/hwTfs48orwnNdW6N5rjwuhqRLsqb107BH4gVjVda9wK04FkflBrdVntyliALXq/RpcJmxFGloDBhlRHWM84v8RMc9KmL3cc24X68TIryKo+iG9Xu1lLPltUolNYtW1qMwo88uff9nLDmEYr/R9jnpONemVGfDZ/UE5xnWHeNL1KvyUtkBNsoxsgUHQumDgAWnDUBQ5Rq+RgTEYqQC88WFgH5jFOuDIxueoCH+mnSEQvkGCA6GIXnw8Pjtnb48vjtmHjyen78/e/XHB3py8OzsHFyyhSteBywY+zM80b8oa+lFgxxSmUYMTJTcLKTM3M02hreM8mr2Mn5I4bsnJ+ds9dvhOvd/Xu15uZ4dD8trTunXWNTDyOiNHbnxzi9iciDSOw/V3lDqsAPFhPIgeen0q3NBBt7sjKRtjydyNXtHhlBxdOXlOVLdUKyMtIGIQY9+RIbQ4S6eGLwXDHIlw2Ira/SJFnGoBqWPFrY1EtVDYOTLa2GXyazdOhk7a3dv2PrzIS+ra85WfXTzPKBkv6jXfpiMvyMvvym2fs//z9G4bCWA8QC3M0Z0T8fQTee/7g4QjZBKBmf4Jiwf481Py6X3y6k1yVsGQ2mBDDtzbVCDM+THa1TJtne8VtJ4FBHWR2kqAIRH6c0w2xKE9gvXllY2U3Lg8WfKVMBBdZbMh0Yd8iRFuwd3j3H12hFHLwpcnIg2OCYI5jjTDsNGO/JPBpfcP22A0HLKGkr6uLXYKoA6quCcMaj/UM3L+6gTF/Zmo/dCenizX1uAUsnXkorE/hXbEYU4wPAZt2YUf0T3xAB3tl1bKYOMB6acWupNozsGYus7BTsu38erJ7O/P0P8q9WEPgcGj+Ocnq2zkSqErlPCSAe8FBlt8bvcjGZjkLwxa7c/AweFDO6JiIXlrHl1ghJXm0tZ96O8avAAcQMN4wnu/0FZRWl6D7si/GAdBgHG3cLpl6jpAJDfoIpuqIQB0QxEJdyJHuCFr6GtF4FE8Nm+0tIK58S9yv+3xGSdoPENP8ULl6mrn7Nw9eKEFzsPW6Zkf1b3XmTo1W+WsNmN4HZMfCf0r7DeBEIfEUTvBeZDors2Mb0L4thc2H8XUh2gUqobH7YvhZOxhcnifEhh+GQy/zA+/fn3727ekq8r7Pgvi4PfuAXBJN1pZQe4LiGyw3fhha/NwFrmHBMB7kskEaokxBBrG3KzFGN6NMNaOW+6iJJ78A1BLAwQUAAAACAARd05dMIPoUMkPAAD+LgAAFAAAAHN0dWRlbnQvZ3JvdW5kaW5nLnB5tRrbjtvG9X2/YsqgNRlz6dW6T0oUIHBdwIUTb2O3SCAIxIgcrRhTJMsZ7kpRFug/9A/7JT2XGXKoy64dNAvYkuZy7rc5M0EQvFutyqJSYq3KRrVarOpWmLUSbS3zy1xu5K0St23dVXlR3YpSLkX46mYiZJWLVzfXURIEwcWqrTciTVed6VqVpqLYNHVrYE1VG2mKutIXF3bsZ11X7vtGmjXvbeBbWSzdxpt+wuwaRGvHv612sXhjVCuXpYrFd7LB2Vi8V//qVJWpC96ks7ZojE6A2KzebOrK7X/19tv371+/j0WrZJ4iKeXFxUWuVmLZFWWeNrC7MWFW122e6kyWaiq0acVMBFXdbiZXV1dBJC6/wcHphYC/YiW81WLmr+QV+EeTqVFbg6CKyqhbFDUQhpK++u+//4MbeBkSV4LQ7pQwNRCO8r8vcrMmka9Vcbs2AUFW5QnsTbFV5VOoBa3ivUUljdJivp3EYgf/ttfweb0QBRNXt8UtLCmZEodYqwFDKwutxD9l2anXbVu3YeDTtOm0EUslnjmxPAOQ4hkR8CyICEqrwG4qEfYgg7d1BlQJdafanWhqs65RMFkrs48xyWEjKxwTd4UuwBYctURkIoIB0g8MGzS9E1L87f2770VZaJOI1zJbi8KoDZO4liBwifYNgmGbPEA7AF0FDj2SslzW2/Q6P5JgIl558pWtEvtBGQ8jIv8BApwv0JiqGlwR11rGEl4VgZne/PDuu5sPoMWRrUbWgEtw11Q3ZWFC+l+nedGy9f5K/hQLGqchsmEUwzwvMjOHkRh9a7FgrYJL/wAOIpAW3TWwTeW8OxZ3sixyME9wShT4pqiKTbcBFWbAq9Br2SgKCdY5aBdwZVBD+8CAsVQByBLA4AdIxgQPj9jSKuiqj1V9Xznq9/T5h/bB2U59r0Emg0OHyKwng0i8AIXxtoeEltitGOlgO1KGUAYqcOKj2uFEGJBJpUWO5PJ3DFb4i7wSv1inhG9aaQ3hDr/Wy59VZnQQDXCtTBC0lQjgHU+fEYIl34l5UwAa0MAeQIEoQCoAyJMJ2zwyZa3Dak2llkAQjP65hmhgBTV1sZRtwYXY+WiULGRBtrOs63KwFaL3voAQVa9WihOFRaSRYWnp9sbBqWoN31mtujcZ0DRExqkY7BL+W4CC9w+9zqwlqsq0BUAoKgcEvVmH0ViRuIxUadeP5e3ImfG65FaZsNdidKg51Jqd/CS1WXWh6TuzhUCjAY44jaNp1V1Rd2jRLIlEKwMKlF0JqjJtaLdF1puPKOwB/GFmPeZT6HRS2NsvSGedZV1L0l1CIBR7BxnnMOwdOSIb3Ye2U9boYEcDoSwlpwfdp2iQIblan7FPWFgsns6/pmvAOk9FsFicC2mvt1nZ5ZBVgULxi2rrSyBOCl13bab8WoXDv6NaGNmCVWjMy8YOcV7iZHNB8D+sD0MlLN/Aci26ytRdtoZxNEfwkCIrIJ8q8MmOECbiW5Ej7BLWyC6HvYX2ZIoQa4q2r96+EZmsQLLguqQFLNlACkJtZWYgxd2vi2w9YuZeQTZRzHueOGHQZ1YqCdCnJ1MBCH2+4HRvNz+1zgDOMrVxD4avaPSjaszRoILUtUtJgMPg2XgMv9KsbnawEpGjCQ2Gj/AHGvBvCalrNICAiypX21gAIRwLug2EOKMQFju9i9cx7IwOgvaYs+czMTkVGSBjV9pIsOoQlvblaYQFDwwwGiowwIhtArAV6Sd5aVFRGHc1CkDdO+qfuSz1LCLfJUrFnpjG9AADg6e6v11d1hhoHGn4m9k/5K5UVYizEYaVP38WsT+9e/sOIsj2s+hV2wZ+k8d0LSavTukD4iFYH9PBK4EnNNRwI5twBXUR5AoifgwArBotJ/ywa5jo2GMgEhClYcXvz+oeSSPdcNgBpKdsS5ZliMelpNAriEFguMRrREHgSnw9Y97xy4Tw80+wMJZJ9PmcWGH+NmYOeWBgczhXAIVXwhGo5y955Jg88OIEHAjKiXAfEIJg6rzYetH0yK1giqyYQxXb7EN0BDurK6hgIVH5gxhIHEaAGx1Nng4ALjjN+wiCIRHX90u+EK+3qs1Q3BjFKVbbtIKk3GEMh+ybdy0mF0ibcCKHghKjvtNIVkosl2IPJuVHjfluoyqqtWKUq3+oW0FOgcMXqBAT3E5s6hwiR6HRnOiEZRMC/lnqU1OnstKQNULkYZSMY9FHS659I2/EFsGel9n04mTqBDUsALsALU8P3ZIz1aB7V4BPB2T9GGrcy3awBiA+nCzckJ1jMxtlol6xjgogGNI36FN3mxCjIJaY8xFGKIgpv+BJEjOL3TkqiqwkYrEfznxcd1jMaLAAHdNeFB+tGTM4SkbeYi5PgObxct9yvdVYAKVYAPnLU0d9MD0QgbeTKU6x3k9PQrHM9JI43stClys4ZqSrooQPROhpwttygiSefbBlJjo5Wi1E3pDCANWMHPO9QpMywVPFZXxkH2xpZO4Yfgycpr/Hc/GMPmI7z8Z/YgFVq+OKqS9IX5Hr4+m43YCT/6Js4M3gKKLaS8J5yZABBJwlsMAzNRgVsKQxPf74048/cRzwz9xcH5wrMB45YvPGPX1AFP9qyMPIUL0SewvNBfhHSoPjrlCfHqnfgiEYzFVsM9AU/CNuY9ffYvAn50Cwh4nd0XImUfpJ0kuQdLw/icIv/87w4UdZ1+JifC5vAvT5VSwmi0FW3MZzCdCyM85+x+hQYnhmRFcTdGLBAy8fWg5JGVB9SlMSEqnexSDNbbrlD6ywcVV89P+nNxxdMTz4DTI7DFm2MWR6S5xQRmuOC4Nj8Rx3MtE9wFtEU+uC2qg+HrQEH4dXqZyRh7d77O3x+anfpUtK67G4nVM3PgTbFZfWpl6I60h8CSxQRqbZHc5aQbrpXXQY4AZQzx8F9fwY1OKAKhBYCNayKaqQRIiWUIg/ims0kysSBws1xjAJbhkdFmaUR2P2zOGExhgsui/ETau0au8G/Y6O8eAtYAJmLTHClKVsACnwoPAAyK1vd5WRWHivtw0aheETdqUAmsbIebnEhdikV5ddhWdyjMrg3F9xM9ki7BsJFhwTgB3cVv3MwVMu6zuVjDzTVcL2x9Vi5D/9qPhmxuY49oJ+Pvag2ZWg9Im14X7P2BI9CLRtwPbcVj8DCS99KicnqZz0VO5O4pgMVL50VO4GKndPU/nSo3IyUGlrq31/GKBPqAdtQx5GeJMrFI4rXDsyfazh+dkVgxXQ/6Fu6G+Z2KjAx/xaB4ifW94X1NiY89lncVCzn41SB+634jYFOp+Vy2JAnui6NXAi2M1KuVnmkurdqbD1sJP4grR9OAZ2ekZIvrxoU8/OIYyXJ8auF9GoysbefpJ3m0aHRDO2iDVeSEqdFcXsrxIMDEI8NiWlqVs9C4MYO/VTPLSwiTi1oowp+CIkr4hkPblSctCuqxzoh9ehpKnRf0P5B6bGBZxX//l5DCISJzcbiDpq+ecKTBVvXbQpsiGY+eUfVGRANRdkmEZ+U9nRl36SmjCXA2VUw1HvcZRgvXOoK99G92Hjug0JtLrTXWmGrLadQHZxmQhV0aehHc70WYinTmaz7fVZENdHIBajG0gvh1mHYfrAhqOIApabdjWAnZ/g/Iias6CunwD1chH19pi38j7l2/fUtJ1ZhyTrKd+Ef2r4anZTurEBKWN7nuwTJntL/AtgGbrXVFQyTkE4bfipK7CFmzdvrbrRAqzYCtPbH/WweBHduL/BtQQfciN1Ne7tgUbh8R6cCssBWkvHDGphq4oYB0a4uZ7J6k5qV4slyFAYcQpqdlxY0Awtxp2wtEec4H8hg2CTOzxO8FyiwQNtSVNCdMA7p8DeAUMqgeM1XQHSZTD+3qmyrO9xyF4G42C2k1Uw3FSNw+ngbs4zlvfwbz32DC+QDwG6VCuYMnUDVsLGsqyNAVHPToSskRmG6FGXgMlVblzTozddIm5/dOxNuPH5yY3PDzZGh+enoQWEoqTsjSKl3s04bQX3awjrXhWO6kvwrCur21KF4TnOAWfdGXy5MiPgloQZ+tUkdu7/Qry8vooOoOP9uwNs3RDgY0lyjZ45JnBVlCVjGHd0yGasl6bgdeUuhfTUlFixNm2dKQ0Z0/rpBn5gV+P0BUosvozFraqw2sWLRXLVs30D8Jg6V3jB0WNJGH0GRe9AQ8+yQ45MflQVGPkMg0AsZJ6nA1r7nmA2jAzmwDynSI7da0cMBHxMpEFjxreAdE1jSXWxLO2gbk8dOaNAxrg/5WmChT/fB611OYRKjllDfV4ZGJr3hO8Ds2toFb9ecVf42EnGz4fYW4JmwW8StgiFaXpYPCws/cxOqrYSRHys5DP84HMVrDUH5s50g14TeFjNkF6gOkUji9a+d9EfLVjWo7ulhDNLJjstS1HWerhHR6mAjZySuSOPFcbfU2AOl58z5JgA+mZKhmDfUXRl2eMAKIT7ua8jqfGSUlZmrCiWzMOih/I4CSM8I1qotBvxU+TabijodDqwOQ+KquloBfk3rBkNEZRq2AKDCT1rmV9OFq7GcpCHGfE1bLKNDnTgMHRr5kmSxGJaLfAcPACNEmxVPd5q8hUOhSsXodi/p04X3RQUW2zMoRmQAO11AonFuiSFMpRBT3RWwgkj9Gc9GsXlxLZ7WDS8gC41+Ks3xwSkUHHemjUtqXwvxUXuAUpBz06IE+4ou/Bw8PQEfYLcBGplr1OKXXiuO6z1czX893tVPdNis7GQ0ZdJ4+EEKgyQEjbicFvUewYu0fTeg9DzfcIRgKGTxutB5ngse0JZbKmYmMCcQFzZxxO0fYXJAxWYHXE1DqLYJQktuTMxiRK8hIjolUvYV4jU5TB9WIKjjzTZ+okHFo3MLVFFPhxZzgSmG5kfWCCI1xoVPfpBg4mwzqzwSomOCHeyLbAgtXoHhDt8oKb9cwp5ChH7WJPMiYq4xOhI1wS80SmJS0040mXroQiln0lVJZ0pSp20Vf8KFLnX/YtR3AC/Ujjbcs03xAK8WzIYq9B0MQJTtXdkKu4+UtvKz9p6D5CqstTeYvevyG7bApSwvrebIBi4HVqZpKsARPhlCN/pjD2+ZCLuo+EVm3us1vPxq09EJP7Ugx91cGS1C73HaAT9GM0THdhVQMtokwYbgujcKoVNL3CBBpt1GLPw8oAfq1mlebxSY01DyfepLPNBaXpgsONHYlYkRzzbcSepMXNY7NkI6NZii9tql08aoe88/qrBbnjh1cGDBa6V8KRLjQzYtoBClxJIeDk5we1iLHg+HeI2TE6eDYcWdMzb0lXRaleowTrsD7D1zZC/gSpq51t5eOYyfQQtOxW4eTgwcYJwupSeXfUx1JeNe/nI537Y2ot3NPNobwKf7rpCeHjCW1T8wKtHRpWThT5+GUkoLv4HUEsDBBQAAAAIAKGNTl3L+aQpARYAAE9JAAATAAAAc3R1ZGVudC9tb2RlbGluZy5web08a3PbNrbf/StQztwZKpEZ23lMRy17J+k6nex12myaZj+oHg4tgjZrimT5sOMq+u/3PAACICnZ3rt7NZNIAoGDg/N+QPY87x+3snh++PlM5GWcZMXlXJyVH1+LRrZdNRdtHWcFjcZFIrIilbUsVlJ0bZZnbSabwPO8g7Qu1yKK0q7tahlFIltXZd3CkqJs4zYri+bgQI390ZSF/ryO2yv9uZb6U5utJUOs4HmeXWhwH3A6PWjvKsBJj78u7ubixzjP44tczsX7uKoI41/lnx0ie8CLgsu67IrEWugfCHh9+PjL+w+f5uKiy/IkqmBq1c7FqgR4rYzkl3hd5bKZCwBVJv3AnOgVNRWQYU5wyos/5KptoraM4qK5lfVcVLWsYqCIpmJUl7fN/GB2cHB29j769PrjT6efRAhnD1awa5ZLxqj2gid5XFx28aWM1rBp/nuQx3eybn4Pfk+e/h74/71oZJ5GcdsW9O1PxPuPr9f8dsNvJb3NvnoK6DqvaPIlnotmdBW/J+VtwZNnHmD3+d2n/cjdZE0XA1IXebm6tpAy+FzffOXN9aZ5Vsi4jtLV8df+4wlvBwz4++mPn375GH14/fHTr7Cn7/EOwVrWl7L25kIPJFJWTRuvriN+FOVZ03qzg3/88/Tn59Hns+jNz2+iX//n3Yfo/S9/++3slKERDQMGgcDydXQl4wTWHRwkMhU3cZ4lSJU/u7hos79IZiMgedm1Pi1esJQBN+N18+IiayOQQUmjsMPPZSFn4vAH+rAgMoFavI2zXKRx04quAAlqRHslBdIyl60UgA1sItoSJIWUSyEFBF+DtIi0On5FyoXQtGqU9erqgEfSITIia6z98UViD48bAH9x14KyFoVRJr24nz6EFw4nFfFaJjCMNGeyBDQU0Uo4VN34sxlNZVrD3KWPUxTlZiIta2ENgEVRUOE8mseeHj0nUEgU1IIuR5QuJViUtubtLVbOmQm0QlMSWL/0Uz0l2CDMrYdb4Sch80baABjFnhw9FccoWyhNUOB8CAI2tA8BfAK7SPgyDsvzmeYpPmAyGDaC8YBJH7sCLeNpXZe1b6y2IrTZX4A8SYKTorkTcQqj2rh7zkbq6A/cSVP1cVspjSK5WRLZJ+ipDvFU7zHiAWIM+lKA6oNB92nhSBsV4WFqv+neozm7eJ9AOYdUZeUEfPXZb+PGAP8OVDXtGnQo2sAHwnOgpt6PV3J1LXIAkBXtt1FznVVKDppwQ4q003TNvqm3Y4CnyiEtxKbHZLl4eb41E5nyt3UJHidRqqz1sAHdIboF9GQ2qZL388MYIIYjvgnZOgUp8L89ftXzwkLjEdw4/VKBMwWpcThhm0lLDNvSNprfjUjG0rmxMBkRjP0AufQ/QQjYvERZskCCAflIDZ5p+Th88ebwHQhj3a1aMByJvMlW4KnjauQQIIjK5ZJ8B/x33vuGM9iJ/EED58zSDE7aCx/tLW6z9kq8OAThFhAs0NHxdMpvPCOLNuke6MwUJdUQhwB310gjHSt1bfmhLlfgjkoIUN6Ac3hdJG/QOfxYFmkGYRPh8fnsbVnDSJKhK4zzn2Qha3KLB0at4aBjAIaRRM2siFBBw091BxJ2UVzQV3a0pLWhV6QvPOsResgOfDExKnSEat7D7md3DcwsO4j8GCZvZHCYVLx79I6XsxYxM8L7qBIgxSGEkmQJZGKooCVpLpzgYkXECmkMJGjfWY14hRvPW4ijLeqVGdWOn52JGbePcX+Ew/MqLRtwZEdWRgfU5+J1MN7VhVBeuYei9AoC/dUVeMm8rGPNBCeuomyDdI20BvmzhC9KX+C0NIFcDWj5xjs9xjDu9IT+f+5th5blc5x32oHx0nUHcdiFFKfHc3F6MkejDgvVoSGikHXRoJk0gfk52D9/aWLhc4NHGNK2rvdWJBg6uQgxtoMlff4ZwouLO78K0g7kFGlERppNcoXrNGbg2piSkNvkdxFhsYOAcyBBcb2A1aifIEcizqurmAe+sqCwgZqPzHpSlxUIxEKQDMK0o+Do5dCOJdmKmDNHiOfGpL2tpfxLiosYaHIrs8urtsEpmBRhItSs6uwCzBxll20MwTs+R7NWVrJAkcGZcHA8/OnJs9PnvXnjZFCmrbZiZyBH2lwBnAifcaLUB8dIBPE9nAA57RMJnMALN+ZRnDTbJ0CEMcEzi7Q4VWUDtuBGKklSgg6U2y3yilm8QGo/F4rnL77dIWEnL0/0sXL0TQx6hj5Xr9+Df+ptCObWbLbRn7bMDoXfXMVxG3uXrToayaT2uUam7cDXINGPBjUk4RlwP7qswRv4b2M4UB+EXsWNE8zLAlN4MNjgACK1khY2ngWcN94912f4LGPw6FJ+wVTAaHag1MqQ+xsm91ScwyxIPchhN2MQ29lXfGCshHng6TAMdJYY7EqqPrSRZeMz6hDFDUsMIDwkcKGS4FSMRZkQJPl8Ik4s14eLlUaH6n2uqaI9oU0k8MFZ3IA/BqAezmyulYf+8fVvv74+i87ee8qnGJGwIkctGOrEE5mR4SGmewFh2Gd7pF07BGfhcMZMApcWA2XdAfagSgrIHIU9TnhYHFI8mLn+hX3LhFfprRP47Y4M65HJsqepgLvsJYHamTxAXLc9GZSi0cCgNDJY/VhaTK8baCjGTxPTBwR4am9UdOuelupUKJyDNS7uYzNblDrY59KOHeGzXYI4n6IPpVkEFZ32pofsTZldCJlsg2Y0xCNoZEY4lYSZTbcGh6xOZByxlh+blZQCueSzgZvj7wFezBn+DmmZNEj7BGiv9NwLjTRw8kRbO8RRxJgrDqjQJDIHhjS9lVZ8QiGEiRtMJqQAUmVmYYuubGOIhmbBqurw/xJi1suu7IYk2aF77CGGun4ffR+rjFt18Ca+kdEK8/yqhJDIicvA5kZJVnMa+ZVK2JiDy4riMYjVumKhK9aGOINa4rj6p8BSia698tVX2+/BIz3pGXgvg97hhrY/epGoBJjnB+trmIuHl8BSla7JLxCIR+V1aIwCk5aObCUDas9nwouTuALamkITnBA54bOyGSGxPTonPgjUn5aimXZcuEU/I6hatY+FAO6HTQZvFtzWGWQ8rfzS+jgSJN26avzNkycwB0w80gGVEt62GKkmcPLwZKYq/cCQ0Ova9PDbvkwMOgG77uS1GXbZPTj4QlyUJaaUHAshq2G5WxmgqIxjY840haK0FSerGBvlqIS5EBjUMk7uKO2GieT4mFv7ywR2YP0BPr83cbQ5kZY0MzKMbvql05miTR5bToDwTdTb4dCKD2372V7BDi4ARwq0sA1JvdslIcyA5HvklNk1vc1y+XPZvkXPowPpddaYct8UMxZi426ho2d8kTwLXSdDNvnu5DlkDRW4rhVl56EHxg/oo0BHZZHfha53pgrLUGF2hTpzRmA2Ycnt8tcQnAGhU00cvt9ogdB9VkUHsCPxyiK+uJZ3z5qruMJ2yI2s40sO/pR+CQljd1iy6KSgot7zk16GVeaIXn/gNPYFYtGubsVEqf6RPsCSLoXbvlzSaCUmP7DKoorBy7PjUp/ZhjkKflOb2PkQSyWaaxBMoEwDqgx6Z88WhxYoFAUzRYE/dKDPdp8h9RxGNrg/xVbfaUzCjfpAZVaFV7ihN6687s0fFA4BGO/1MGmgOgSjTIWW84DkiKjDz0wQwU/uCTvt07BI6uMQetwzArUebbkVNw1ou7ubre6PjcwHj1dldRc5J21Ln4t7oR0o4YAuIFrjXNzXoVmDkS/Ewj76k3X2l6yVMrdlG+cRusCGIhJFLq4fk52iFUFe9zDqvgoTry+S+Owj+4rbuF5j4QJs2Bf/GCIbNJy+BR+yUyon8YmpLNjlbVblmaz9PiQiK0KkcfiOz8X3apMBS9mUEQjxVBzPwDvwPDt9u6yxAwvYZYV/HBzN1fxDNRUXKcRtlPvHlj7wdkfBSzgQgoI98SIDMKzx6UOVwRO95cyxt5pkhg9ziwqaXeRe3L6zLqhqtpGfw9sEi/6aw3Jkkc8xQsfKwoS1tot/k0HqqizrBJge57JvghRlvT4+OjryNAZpDYYdfJWpGCJtLdDZGox7A5iWTlCEwFDKPXtuf0sCn2ChOs/auwnU3SrmztwCPMbHrqAmy0+/vfvbKbVPnv0EOc2veChsVJYVpzxgciEQsAOMKu8aQKhrr9IuxxgWktI2JsUcRFM1QCjXe+KrD+/O9JN3SI0D22UcgVC7pMR65PE+FzKYriuRYDh9EOvjc2WCjIhgg6qlcp8mKhqjqaspvr3IEgClshZbAALS3J/kGZZch1s6jWdLfO87KIZbdMkGS15yXbV36nyNpLATg3AWcs4ycBjE88WJVjxiT4DDPv6niEOmbR0XHaq6+wy9DD1edUkcQHQa38QZ3S3yRykLTbHARHGeW6BULZearozk0kNb7p33yF1PnAGHPa27PIT1DBjabGdqCnw5fjVTh+Sa4BgQjd8DSc/RBUQNU0ftusgy6jzM9R0xXKkKllatctxdGL50bZId4T4c1UwYYg8y6FWBZ7yWBVrToIoTzNyiJkvQQHg1xs66ixCvosI4KCQW1oWMMM7g/K52aSECIQy1KH2kN4fLua7hU2PRAqhXQ3TQpSlGLmruaKX+uFwwnueqegmBCBzmLznBXfMQCKPlPV6tuvXEZCru0UPUDs3j+ItydOMF/TOY//zoaNYrh4XT9+IYdZ33VF8MzO/32zEDZy4MdtwSMkB2NFosv6JTU2todtDHGLSLdMJIMGuy1sxyGYKv2yvI/AQmWoM4w2IjgRiXoiBa4zQBROVSojlGAePJc4twE3VcvJyIpoImLwnQgsE9tRaOrhYJfT8Re5fjp4gUCCKiRBuM96UT41UD8k0B9uJsSkJYBOuXHg1Rkgq2S8R4n2AaVs8cFKk1Vu4gmQNx+vjTG2+UbdGLL0vi0YcXKH3aWg17545Lut+8DF4E6zZL8ARz/nZFmbU2xsOXpmyAKTKEsu79T78ytyjovO4FUt/2nnN1xtl4o7tM5snosqlvbp3uMHIRfaP2O9tCUysHQahM4XrcpZsoW7Pg9DFpX6Pg2P91Eq//afIqQCqvp4w2+qQTefgC80suWkSJXMV3IRhtZe/6FCKcyknmRu/1/Di30AGaBCZ+8z30vh6W69A5J+FOr72z34hnz2TRWhU9dB4McNxy3Dvd3/H0GjKIyybceHhTpZYwAYxD6y24/Le1q6qKmjhxFQMUXSO0pnBqMHP5FcC/khiJCT3qD7bwrFoRZNJ13NKlDvzo90ZR3ZcsmwZljewN39QDFcGrWQGExim3b2DZvxgfgZRR5zPGS7vrsr6jypIuvuwrZWPcHYoNX+HAUi0HHFN9G45TluNHTqYjPH1thZZY3k4Pz/dd9ZoNYE03mzQqk48H6Ew0ifTy0aNzfXKriL6Y7ly6m1A8uRAcoqmAb8GxWg9SRzzwwB1wQVlmjVA19ligAVB54E674MCy4peFsKMBO1hZCBUWWAHJwtgJFyIF/wtKDbC+r1w75zgRezXVDOzdvguANmarQzfGKlit9zxyp05mPjDN/srVwaZboQmHRLIPto5UA7No5KrDsCbCOijQ7tY85ThktPh7c3ijbVTLCMeTn4rjfg7eSwDPFLG2u/ECuovIBC5E8kGQQgyCVQW2U7RBmU1O2VxLyNupfotFK7ZcXKma2WaYZqAElt6MbzHQyLRnT7ncODd1YdpOVwkRLjymmxyQ26bZlwj4fAnefuvAo1hHuZOuLVcxxIDakUzdwnuAdxkHQkhiTDSo8P7kCSE6C2j0GcvzsPFIWTHtkDUpyFUrfZw+AVt1JxA9kL4P6GZ0JROs/iEvZgzi1vLpJCLU+9sOAjGW94DeeNfgIl6h30oGBXJXhHRgxCqPY33fFjMpFqKDwS5dQR8i4/XNlDJXyoeizNOxicGImWkXoKPa2VJmygV7VfnEJ6ObXhPkdVfMAszc93fIRzegTEjk3H1w8dPbqwOB7CiJGradHu7OHycKGpWd4sA6YRXGkqwhDD3nWHgI52gu+kzeogjwp2FNsMqzim+UYNEwcsLHYx0P2tIHmEzJhBabChtJFoOUdTaSYhOWiD6WILCdvYg5xWWG9X9lx7Q5fxpaNpi41hXLPY7mfLyiqjE5T9nlLGxO6W2+28FZ5BlHb4mYut2Wej4r3KYnzOJye/jDZkw8GJ/vgGEfHA8D0e4kLbYzMLJp3jVXE9QDPkwT8Ae8xfoIOzipw6lnUU3LzvEre0tt3XbbzeEvC/QLry5pmAoI/VJEU74t0ZYavz2GMhIkOFMndxqV/5jhGBcQHm9JIIK3RZTlq4ZUj+NLZwvL9j/qEPsjJ3yNQyFbq5QDg1AGHZjr1mzbo9JVtk6ONeSIK+SaF335L3F8hCNHLndo++gGf1kTCt8pG3/R+RD4Hmz6k+l6BmBOXjx58nyPL9qXgnEgdWSbWIOGCkzwHhofdnlIbnpxjhtjYDwc3meILM4/2/TSvRVrGRe0VZRDeMWBfLgxGCyCl+l2hy2ReVw1eEs+K8KNP5GDYqOa81RE+dXRIjjZCYxyTiR9dHmhEcCvtGaHLdK8RXYyP3sOh+FU7I0vvHSkXNTGvleEaYtLDACqLgJqSsPImPw8uLvIpbI2oFLXUlLzEFLtAWdTClPAf4OYPkhIt4NQdOoqXX+Lbq5oCtTu6wPLASXOScvup4VZ7xz9/N+npI/RUnXAh17C883Vvvvvu9Flt72324gU3KCGyaoJDUkUdfj2taEf0ICm3t2ouzxwd3s7zaYyoH/U8oILhYW85TJoox+cvHzlgi6LiJYv+r8PsFzSj4tMwxh/ZoSIYq4w1Vqemt43mNUPwCS1O2uQQZicrZCdVdf2OapsvhNNDtmv+oUsNqn49hhFCliTvueO3s4estuMeVi7ZVdP5SH9tFymqp1mt156Js8Cll0ND88X7i6MW4VNCWm9qQgC9awKhalL7qpJWll9/zcq6M6XnWTt7tBQr25Pg0Y3Z3DeA3sz3Pho+K9fjJsz9zdmhk2Zf6kb8/BOjIOvzuyXG68uqdKH1Wv6OwwYm0qqXy8nd9x4WETBFbQ3LuEPC0YGkggzBU0VzqD3hRKY7fl2EHbQb3waurSgJZS70aurGCJEiN2wezIOXp0z4cUelmplV0Gyo8v+B5xKPAc/H9Uvto4RmgG1Wo0ALQCdJvSq1hsv6/E1XQClUHBahOMWasWOY/9H6mkTBTWy97zrTNfVthMoBVVZ+R53oJCVUZY0zh9gsM6PcTVZOGox42oQX/rNFi5Sd+mWx652/L8U6RTv+dd66gcDPOI/ecKoDv1M6H4FZMqooU6dfXFZvxKJDcPEkVs2F/zE7zFYLuYOrRboOPEHy/Tr8DjX248zOYZnWcyNMg7U4bCMBXzH/kEd31In4Ha7Qyos4YDVeLmDjNRfWeWToZrrYw3+zIVCQadUDl6jRF+7ZbtON2aQnrUPmM5IVLSy4N8u6qmQlvTmfavuDFCrZG+AuAiO020zoc5ilDCoqEntd/C/UEsDBBQAAAAIAKcKTl3VcylafQcAAFUTAAAUAAAAc3R1ZGVudC9yZXBvcnRpbmcucHmlWOuy27YR/q+nwKA/TPZItKQ6mViJMmMnrcedJPU4nvyoqnIgEpTgQxIMAB5JljWTd2ifME/SXQC8STp2O9WMeAF2F7vfXrAgpfQ7xZnhhD+IlJcJJw9C1ywXH5gRstSElSnRdVEwJbgmmZIFYYkBCqITqbgisjZVbXREKR3Z6TjOalMrHsdEFJVUBmSU0jh5o5Efe69l2Twr7jgrZna52DRsb+DVTZhjJcptM/6iPI7Jj6zCsTH5mf9ao+IjR/rm9Q8N3euCbfnY3b5XbO8pdKJEBQrnbJPIopAladVgaYx65UNCa2hDVDGleeyMdmTRVsm6THsKllIVs+l0GhsZV+LA89FolPIMFihTruJK8VQkFo5AV7kwMS8NwrtojVl581baqDEavF6PSY9vQQYEcFmPR+T2L2WGxalQCyQjHy2sY3Tb9WAipUrd0JLQxgz6qOhcFMIsiCgN0H85BvsB6likfUtQN1jgJ1lyIMJbSCbfAqs2K1x1vbDSIXzeWnhIwUyy4yl59Y4EW8V5GdogtMCnfRBIAM/h10Tfi4oUQmvA4+mGpYOQRNkic5aBYwwoS0490wi1HqLnRWukYkJz8gvLa/5npaQKqOMuam3IhrfuJVIRxxw2y1hAyDdk9ilpjqiRVkktjHjgXkjjrdZDABrCFDQT4di9+2nHpXnOE8NTxB4YNDdB64sQFWvfiNAWBusPnoNq+GSFeIlRcQ/XAPCGsNTLd6qGJOIH8Fcs7+2rX5M9gD+WZLW2rxmggXF8RISHYd1iAYoMNO3rgj62/CvaKEvXjcf6XItBNCayNKKseQ/vPSjVC5NoC2hcSQ5bBh9Yy0FqByDGAofiHofMG+VFAB3OfEZBpwPWOliz8Sp5OjQep+m65dkLoLaFLJIVL4OOCJJDEy1rlVysa0kwFuxcBGo8cGUC+vbVS9oZnzq42hoZ4cWJD3urp1gddlxsd5jpdjrS4kNnFHpfbt6jr7wd8AYu03Q9VOuQjMkR/ps9/BGBglVBlktmxihgRY8yl3334G8jD0B5VVaDIDgkZAKyAL55SP5IcHpMgqMd3Q1GH6thwx8KvLsp8O5CIOThAJehxohrpAAAVm5zHoABNqFzUfIl/cN0ms6/ZNRLWM5v8Bp+MAHyraZQ+gt2CEAPfJ2twbbZPIT1Mwo18oSoPYHNjOdP1meQmYk879YIb7nIxeuFY2bgGPgf5nCfA9zWGxtYsheHNrCaarpcNsVzcQWu81jg9S5EGXiw7F4ZHGYhGtCbdiA280c7/5jLHpU6/4xUnB9ijen8mPY34m0I0v8TAln2bPps+r+EQGuO9X8bDHfEx8IbKHmPRoNfrltFswwLIZipIJXrTaDo6p8vJn9nkw/TyfM4mqzvcHOMqe0ubtTPTlTCKiy0TRmJSr53ZQYSZ1g67sjsK9CV7nfCcHolIaqYNtxVH+AFHwJ5R3VRpDxT6HF6NiZzi8PJ6fqk0RVwgP7j1bsFsd0EPHebwwKsT1uUNjlL7ntqGaZg78BU8FvxUxTvkTtH76stvTIBd8XAMY7Jr9hJm+Py+bwPPGybEXRv0O14wrC/l+RQ3i1NSL5d+g5rWAyhT70fXVN/Y0PEMozdeH8TDsNPtCQZhab36PtTjKJO7Jk8gA2DvutrIITmDlyVEmZgKfAaOX168bOHVXE4GpQOBN8VNyeP2B0zjgFuaxcdpO9Q17Z1hPe2Z3xZixy0gIpUVHAugdxX96ncl+C7Tc6J7+Ohf4+qI/nrz3/7CV3NwYRSomFgBWHY3JcZV2Bh2zJilmIntaIfydsag+YvM7h89/r5F/j85qnbXiGWpNlJWMlOJwoCyD0WrOzGDSBeWoWg8rJCw9Avb1/8SF69REroDgws9vGyz6YfJ5MJ/hf/9YV2vZhtMGytRzS7DR8aDrALjzkR7LqpDmw3aVuJyJ6AbDoB8BKPNEtam2zyFe2lYQVHmCRnWvv2xfZXtB2FbDqdO+qC2/U6Qhy4oEnEgCKbxYl4/gUQrbCdGtumqtcRYNBkhQms/xZ4OhqGRRvmLtbo77/9m2KuWPqmSXONHKSzkxL9KTtbIgGnCA0lO+FOPtQhON5AhcAWJQwdG1ZEO9spZQOmSWsImlPJCo6F55TN7C2X599/+9dpJ9xgZW/umthrYa9tqNi3B4gWNwlawQONwLVwQAoGhuJSyw4/fKX+lOD8CqlaYG2cLRG3ni9EoqRzRgM9vdx3c2mZEgG7UFNxEgHFiUy7ZhiE70RDN7ugm/XpBrKz6kKhrIoxkGxyoSbEEbTB5WPN5dxnFE9usdoMvWYkxS1in8OfWQd9ZLkxsptkYPcxjsfbjRMP7rugsY6OffJfCW3D4BYXjseuktC2o2mi/R8ljd5LW4whIkNfZDHVBx8eMDD6h3+bQjjbfVHoHczhUO0+90xsoTUCaxmW0x8I7OXwrDh2+/gNJK2h8if4Sen1990RXHFd57iVns5thVJyjwWq++ziorV3XPQ7ODYpct/vPvobZktlZeEyF2XgerfrdOwgsSqdGmHnXhvghK6aqXWjDjQhXhOPvSMc/QdQSwECFAAUAAAACACnCk5dFq3noS4AAAAuAAAAEwAAAAAAAAAAAAAAtoEAAAAAc3R1ZGVudC9fX2luaXRfXy5weVBLAQIUABQAAAAIABN3Tl2xxTOt2wYAAKkUAAAOAAAAAAAAAAAAAAC2gV8AAABzdHVkZW50L2NsaS5weVBLAQIUABQAAAAIABF3Tl0wg+hQyQ8AAP4uAAAUAAAAAAAAAAAAAAC2gWYHAABzdHVkZW50L2dyb3VuZGluZy5weVBLAQIUABQAAAAIAKGNTl3L+aQpARYAAE9JAAATAAAAAAAAAAAAAAC2gWEXAABzdHVkZW50L21vZGVsaW5nLnB5UEsBAhQAFAAAAAgApwpOXdVzKVp9BwAAVRMAABQAAAAAAAAAAAAAALaBky0AAHN0dWRlbnQvcmVwb3J0aW5nLnB5UEsFBgAAAAAFAAUAQgEAAEI1AAAAAA=='
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_payload))) as _archive:
    for _member in _archive.infolist():
        _parts = Path(_member.filename).parts
        if not _parts or _parts[0] != 'student' or '..' in _parts or len(_parts) != 2:
            raise ValueError(f'Unexpected payload path: {_member.filename!r}')
        if _member.is_dir() or not _member.filename.endswith('.py'):
            raise ValueError(f'Unexpected payload member: {_member.filename!r}')
        _target = (_root / Path(*_parts)).resolve()
        _target.relative_to(_root)
        _target.parent.mkdir(parents=True, exist_ok=True)
        _target.write_bytes(_archive.read(_member))

importlib.invalidate_caches()
print('working directory:', Path.cwd())
print('student package:', (_root / 'student' / '__init__.py').is_file())
for _package in ('torch', 'transformers', 'peft', 'bitsandbytes'):
    try:
        _version = importlib.metadata.version(_package)
    except importlib.metadata.PackageNotFoundError:
        _version = 'not installed'
    print(f'{_package}: {_version}')
try:
    import torch
    _cuda = torch.cuda.is_available()
    print('CUDA available:', _cuda)
    print('GPU:', torch.cuda.get_device_name(0) if _cuda else 'unavailable')
except Exception as _error:
    print('GPU check failed:', type(_error).__name__)

_meta_path = _root / 'data' / 'splits' / 'meta.json'
DATA_READY = _meta_path.is_file()
print('DATA_READY:', DATA_READY)
if DATA_READY:
    _meta = json.loads(_meta_path.read_text(encoding='utf-8'))
    print('dataset_version:', _meta['dataset_version'])
else:
    print('Run the notebook data preparation cell, then rerun the CP1 cell.')
print('CP1 student package sync complete')


In [ ]:
# Chọn một cách chuẩn bị dữ liệu. Với zip/source không cần Kaggle Secrets.
DATA_MODE = 'kaggle'  # 'kaggle', 'zip', hoặc 'source'
DATA_PATH = ''        # zip: đường dẫn zip đã upload; source: thư mục images/ + labels-YOLO/
if DATA_MODE not in {'kaggle', 'zip', 'source'}:
    raise ValueError("DATA_MODE phải là 'kaggle', 'zip' hoặc 'source'")

if DATA_MODE == 'kaggle':
    _username = os.environ.get('KAGGLE_USERNAME')
    _key = os.environ.get('KAGGLE_KEY')
    if not (_username and _key):
        try:
            from google.colab import userdata
            # Lấy cả hai trước rồi mới cập nhật môi trường để tránh trạng thái nửa vời.
            _new_username = userdata.get('KAGGLE_USERNAME')
            _new_key = userdata.get('KAGGLE_KEY')
            if not (_new_username and _new_key):
                raise RuntimeError('Secret Kaggle đang trống.')
        except Exception as exc:
            if exc.__class__.__name__ in {'SecretNotFoundError', 'NotebookAccessError'}:
                raise RuntimeError(
                    'Không đọc được Kaggle Secrets. Trong Colab, mở Secrets (biểu tượng chìa khóa), '
                    'thêm KAGGLE_USERNAME và KAGGLE_KEY, bật quyền truy cập cho notebook rồi chạy lại; '
                    'hoặc đổi DATA_MODE sang zip/source và đặt DATA_PATH.'
                ) from None
            raise RuntimeError(
                'Không tải được Kaggle credentials. Kiểm tra Colab Secrets KAGGLE_USERNAME và '
                'KAGGLE_KEY, quyền truy cập notebook; hoặc dùng DATA_MODE zip/source với DATA_PATH.'
            ) from None
        os.environ['KAGGLE_USERNAME'] = _new_username
        os.environ['KAGGLE_KEY'] = _new_key


## 1. Dữ liệu: tải và chia split

In [ ]:
# Kiểm tra dữ liệu đầu vào rồi gọi công cụ chuẩn bị dữ liệu của đề.
if DATA_MODE == 'kaggle':
    _prepare_args = ['python', 'scripts/prepare_data.py', '--kaggle']
else:
    if not DATA_PATH:
        raise ValueError('Hãy đặt DATA_PATH thành đường dẫn zip hoặc thư mục dataset đã giải nén.')
    _data_path = Path(DATA_PATH).expanduser()
    if not _data_path.exists():
        raise FileNotFoundError(f'DATA_PATH không tồn tại: {_data_path}')
    if DATA_MODE == 'zip' and not _data_path.is_file():
        raise ValueError('Khi DATA_MODE=zip, DATA_PATH phải là file zip đã upload vào Colab.')
    if DATA_MODE == 'source' and not _data_path.is_dir():
        raise ValueError('Khi DATA_MODE=source, DATA_PATH phải là thư mục dataset đã giải nén.')
    _prepare_args = ['python', 'scripts/prepare_data.py',
                     '--zip' if DATA_MODE == 'zip' else '--source', str(_data_path)]
subprocess.run(_prepare_args, check=True)


In [ ]:
from pathlib import Path
from PIL import Image
from student.grounding import (
    PROMPT, build_prompt, collate_examples, draw_ground_truth, encode_example,
    load_split, objects_to_answer, validate_session_disjoint, vision_token_count,
)

splits_dir = Path('data/splits')
train_rows = load_split(splits_dir, 'train')
val_rows = load_split(splits_dir, 'val')
test_rows = load_split(splits_dir, 'test')
assert validate_session_disjoint({'train': train_rows, 'val': val_rows, 'test': test_rows})
print({name: (len(rows), sum(len(row['objects']) for row in rows))
       for name, rows in [('train', train_rows), ('val', val_rows), ('test', test_rows)]})
sample = train_rows[0]
image_path = splits_dir.parent / sample['image_path']
display(draw_ground_truth(Image.open(image_path).convert('RGB'), sample['objects']))


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Dùng hàm đã kiểm tra trong student/grounding.py.
print('prompt:', PROMPT)
print('gold answer:', objects_to_answer(sample['objects']))
# Ablation pixel phải đổi đồng thời đáp án và prompt:
# pixel_answer = objects_to_answer(sample['objects'], 'pixel', sample['width'], sample['height'])
# pixel_prompt = build_prompt('pixel')


## 3. E0: zero-shot

In [ ]:
# E0: model gốc 4-bit, inference batch 4.
!python -m student.cli predict --zero-shot --split test --batch-size 4 --out work/e0.jsonl


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Chỉ nạp processor ở đây; CP2 không cần giữ model 4-bit trên GPU.
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
processor.tokenizer.padding_side = 'right'
example = encode_example(
    processor, Image.open(image_path).convert('RGB'), PROMPT,
    objects_to_answer(sample['objects']),
)
print('prompt tokens:', example['prefix_length'],
      'supervised tokens:', int((example['labels'] != -100).sum()),
      'vision tokens:', vision_token_count(example))
assert vision_token_count(example) == 220, 'Ảnh 640×360 cần 220 vision tokens'
answer_ids = example['labels'][example['labels'] != -100].unsqueeze(0)
print('supervised answer:', processor.tokenizer.decode(answer_ids[0], skip_special_tokens=False))
batch = collate_examples([example], processor.tokenizer.pad_token_id)


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Inspect module counts, then release model memory before subsequent subprocess training.
import gc
import torch
from student.modeling import load_qwen, matched_lora_modules
_setup_model, _ = load_qwen()
for _setup, _expected in (('E1', 252), ('E2', 252), ('E3', 348)):
    _actual = len(matched_lora_modules(_setup_model, _setup))
    print(_setup, _actual)
    assert _actual == _expected
del _setup_model
if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()
else:
    gc.collect()


## 6. Vòng huấn luyện

In [ ]:
# Set True and mount Drive to copy each completed run before starting the next one.
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
import json
import shutil
import subprocess
from pathlib import Path
for _exp in ('e1', 'e2', 'e3'):
    subprocess.run(['python', '-m', 'student.cli', 'train', '--config', f'configs/{_exp}.yaml',
                    '--out-dir', f'work/{_exp}'], check=True)
    if BACKUP_TO_DRIVE:
        shutil.copytree(f'work/{_exp}', f'/content/drive/MyDrive/day10/{_exp}', dirs_exist_ok=True)


## 7. Suy luận và chấm E1–E3

In [ ]:
# Suy luận từng checkpoint và chấm bằng scorer của đề.
import json
import subprocess
from pathlib import Path
for _name in ('e1', 'e2', 'e3'):
    subprocess.run(['python', '-m', 'student.cli', 'predict', '--checkpoint', f'work/{_name}',
                    '--split', 'test', '--out', f'work/{_name}.jsonl'], check=True)
    _run = json.loads(Path(f'work/{_name}/run.json').read_text(encoding='utf-8'))
    _meta = [f'{key}={_run[key]}' for key in ('trainable_params', 'peak_vram_gb', 'train_minutes')]
    subprocess.run(['python', 'scripts/score.py', '--pred', f'work/{_name}.jsonl',
                    '--name', _name.upper(), '--out', f'report/evidence/{_name}.json',
                    *sum((['--meta', item] for item in _meta), [])], check=True)
# Đổi --split val để gỡ lỗi trước khi chấm test.


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 rank ablation: select your strongest baseline from E1–E3; change rank only.
BASE_SETUP = 'e1'  # choose e1, e2, or e3 after inspecting actual evidence
import json
import subprocess
from pathlib import Path
for _rank in (4, 64):
    _name = f'e4_rank_{_rank}'
    subprocess.run(['python', '-m', 'student.cli', 'train', '--config', f'configs/{BASE_SETUP}.yaml',
                    '--rank', str(_rank), '--out-dir', f'work/{_name}'], check=True)
    if BACKUP_TO_DRIVE:
        shutil.copytree(f'work/{_name}', f'/content/drive/MyDrive/day10/{_name}', dirs_exist_ok=True)
    _run = json.loads(Path(f'work/{_name}/run.json').read_text(encoding='utf-8'))
    _coord = _run['coord_scale']
    subprocess.run(['python', '-m', 'student.cli', 'predict', '--checkpoint', f'work/{_name}',
                    '--split', 'test', '--out', f'work/{_name}.jsonl', '--coord', _coord], check=True)
    _meta = [f'{key}={_run[key]}' for key in ('trainable_params', 'peak_vram_gb', 'train_minutes')]
    subprocess.run(['python', 'scripts/score.py', '--pred', f'work/{_name}.jsonl', '--name', _name,
                    '--out', f'report/evidence/{_name}.json', '--coord', _coord,
                    *sum((['--meta', item] for item in _meta), [])], check=True)
# Or choose ONE alternative: data_size (--train-fraction 0.1/1.0), or coord_scale (--coord pixel/norm1000).
# Optional E5 GRPO is described in GUIDE.md.


## 9. Bằng chứng minh hoạ và bảng kết quả
Chỉ tạo hình và bảng sau khi đã có predictions/evidence thật.


In [ ]:
from student.reporting import evidence_summary, load_predictions, render_predictions
test_rows = load_split('data/splits', 'test')
preds = load_predictions('work/e3.jsonl')
# Optionally set six image IDs after inspecting the predictions.
selected_ids = None
figures = render_predictions(test_rows, preds, 'data', 'report/illustrations/e3',
                             limit=6, image_ids=selected_ids)
print('\n'.join(map(str, figures)))
print(evidence_summary([f'report/evidence/{name}.json' for name in ('e0','e1','e2','e3')]))
